# Responsible AI Toolbox: RAIInsights and ResponsibleAIDashboard (Disabled cohort, E1)

This notebook is the rubric-9 deliverable ("Responsible AI analysis, RAI Toolbox", 30% weight in `PROJECT.md`). It builds a `RAIInsights` object and explores it with `ResponsibleAIDashboard`, covering **Data analysis, Model overview and fairness, Error analysis, Feature importance, Counterfactuals, and Causal analysis**.

## Clinical framing and why the task is simplified to binary

The long-term motivation is EMG-driven prosthetic control (see README). A prosthetic hand controller's first, most safety-critical decision is not *which* of 18 grasps was intended — it is a binary gate: **is the wearer attempting a movement at all, or resting?** A false "movement" during rest produces an unwanted, potentially unsafe actuation; a false "rest" during an attempted movement means the device fails to respond. This binary "movement-detection" framing is also what makes the Responsible AI Toolbox's counterfactual (DiCE) and causal (EconML) components computationally tractable on a laptop, which the full 18-class action model is not. The 18-class action-recognition results remain in `01_unified_exercise_frequency_classifier.ipynb` and `results/metrics/EXP-002-amputee-e1-logreg_*`; this notebook is a complementary, RAI-focused analysis of the upstream gating decision, using the **Disabled cohort** (`Database/Amputee/`, Ninapro DB3, Exercise 1) only.

**Target:** `is_movement` = 1 if `restimulus > 0` (any of the 17 Exercise B actions), 0 if `restimulus == 0` (Rest).

**Input:** classical EMG features (MAV, RMS, waveform length) per channel, extracted inside the training pipeline only (no test-set leakage), per `AGENTS.md` feature-engineering rules.

**Cohort formed for the dashboard:** per-subject cohorts (at least one required by the rubric), so a clinician/engineer can ask "does this fail for a specific person" rather than only "does this fail on average".


In [ ]:
pip install responsibleai

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.io import loadmat
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

from responsibleai import RAIInsights
from responsibleai.feature_metadata import FeatureMetadata
from raiwidgets import ResponsibleAIDashboard

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd()
if not (ROOT / 'Database').exists():
    ROOT = ROOT.parent
DATA_ROOT = ROOT / 'Database'
COHORT_FOLDER = 'Amputee'   # Disabled cohort, physical folder name unchanged
EMG_CHANNELS = 12
WINDOW_SAMPLES = 200
STRIDE_SAMPLES = 50
MIN_PURITY = 0.80
print('Disabled-cohort feature dataset root:', DATA_ROOT / COHORT_FOLDER)


In [ ]:
# Step 1: Data characterization + windowing + classical EMG feature extraction (MAV, RMS, WL).
# Extraction runs on raw EMG only; scaling/fit happens later, inside the training pipeline.

def load_e1_recording(path):
    data = loadmat(path, simplify_cells=True)
    required = ('emg', 'restimulus', 'rerepetition', 'subject')
    missing = [key for key in required if key not in data]
    if missing:
        raise ValueError(f'{path.name} missing {missing}')
    return {key: np.asarray(data[key]) for key in required}


def extract_features(recording, path):
    emg = np.asarray(recording['emg'], dtype=np.float32)
    labels = np.asarray(recording['restimulus']).reshape(-1).astype(int)
    groups = np.asarray(recording['rerepetition']).reshape(-1).astype(int)
    subject = int(np.asarray(recording['subject']).reshape(-1)[0])
    if emg.shape[1] != EMG_CHANNELS:
        raise ValueError(f'{path.name} expected {EMG_CHANNELS} EMG channels, got {emg.shape[1]}')
    rows = []
    for start in range(0, len(emg) - WINDOW_SAMPLES + 1, STRIDE_SAMPLES):
        stop = start + WINDOW_SAMPLES
        block = labels[start:stop]
        rep = groups[start:stop]
        counts = np.bincount(block, minlength=18)
        label = int(counts.argmax())
        if counts[label] / WINDOW_SAMPLES < MIN_PURITY or np.any(rep != rep[0]):
            continue
        window = emg[start:stop]
        mav = np.mean(np.abs(window), axis=0)
        rms = np.sqrt(np.mean(window ** 2, axis=0))
        wl = np.sum(np.abs(np.diff(window, axis=0)), axis=0)
        row = {f'ch{ch + 1}_mav': mav[ch] for ch in range(EMG_CHANNELS)}
        row.update({f'ch{ch + 1}_rms': rms[ch] for ch in range(EMG_CHANNELS)})
        row.update({f'ch{ch + 1}_wl': wl[ch] for ch in range(EMG_CHANNELS)})
        row['action_id'] = label
        row['is_movement'] = int(label > 0)
        row['subject'] = f'S{subject}'
        row['repetition_group'] = f'S{subject}:rep{int(rep[WINDOW_SAMPLES // 2])}'
        rows.append(row)
    return rows

paths = sorted((DATA_ROOT / COHORT_FOLDER).glob('*_E1_A1.mat'))
all_rows = []
for path in paths:
    try:
        all_rows.extend(extract_features(load_e1_recording(path), path))
    except Exception as error:
        print('skipped', path.name, error)

feature_df = pd.DataFrame(all_rows)
feature_columns = [column for column in feature_df.columns if column.startswith('ch')]
print('Disabled E1 feature rows:', feature_df.shape, 'movement share:', round(float(feature_df['is_movement'].mean()), 3))


In [ ]:
# Step 2/3: subject-held-out split (deployment scenario = unseen prosthetic user), fixed seed.
rng = np.random.default_rng(SEED)
subjects = feature_df['subject'].unique()
rng.shuffle(subjects)
n_test_subjects = max(2, round(len(subjects) * 0.25))
test_subjects = set(subjects[:n_test_subjects])
train_mask = ~feature_df['subject'].isin(test_subjects)
test_mask = feature_df['subject'].isin(test_subjects)
train_df_full = feature_df[train_mask].reset_index(drop=True)
test_df_full = feature_df[test_mask].reset_index(drop=True)
print('train subjects:', sorted(set(subjects) - test_subjects))
print('test subjects (held out):', sorted(test_subjects))
print('train rows:', len(train_df_full), 'test rows:', len(test_df_full))

# Step 4: pipeline-only scaling, class-balanced logistic regression baseline (matches EXP-001/002 style).
model = Pipeline([
    ('scale', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)),
])
model.fit(train_df_full[feature_columns], train_df_full['is_movement'])

# Step 5: held-out-subject metrics before any RAI tooling is attached.
test_predictions = model.predict(test_df_full[feature_columns])
print({
    'accuracy': accuracy_score(test_df_full['is_movement'], test_predictions),
    'balanced_accuracy': balanced_accuracy_score(test_df_full['is_movement'], test_predictions),
    'macro_f1': f1_score(test_df_full['is_movement'], test_predictions, average='macro'),
})


## Build RAIInsights

`subject` and `action_id` are retained in the train/test dataframes so the dashboard can filter cohorts and label errors by action, but they are declared as **dropped features** (`FeatureMetadata`) so the model never sees them as inputs — `subject` is an identifier (using it would leak who-is-who, not muscle signal) and `action_id` is a finer-grained version of the target.


In [ ]:
rai_train_df = train_df_full[feature_columns + ['subject', 'action_id', 'is_movement']].copy()
rai_test_df = test_df_full[feature_columns + ['subject', 'action_id', 'is_movement']].copy()
# Keep the dashboard responsive: sample the (larger) test set rather than passing every window.
if len(rai_test_df) > 1500:
    rai_test_df = rai_test_df.sample(n=1500, random_state=SEED).reset_index(drop=True)

feature_metadata = FeatureMetadata(
    identity_feature_name='subject',
    dropped_features=['subject', 'action_id'],
)

rai_insights = RAIInsights(
    model=model,
    train=rai_train_df,
    test=rai_test_df,
    target_column='is_movement',
    task_type='classification',
    categorical_features=[],
    feature_metadata=feature_metadata,
)

# 1) Feature importance / model explanation
rai_insights.explainer.add()
# 2) Error analysis (which subgroups/feature ranges the model gets wrong)
rai_insights.error_analysis.add()
# 3) Counterfactuals: smallest EMG-feature change that would flip the prediction
rai_insights.counterfactual.add(
    total_CFs=10,
    desired_class='opposite',
    features_to_vary=feature_columns,
)
# 4) Causal analysis: estimated effect of a treatment feature on the movement-detection outcome
rai_insights.causal.add(
    treatment_features=['ch1_rms', 'ch1_mav'],
    heterogeneity_features=['subject'],
)

rai_insights.compute()
print('RAIInsights computed for', len(rai_train_df), 'train and', len(rai_test_df), 'test windows')


In [ ]:
from raiwidgets.cohort import Cohort, CohortFilter, CohortFilterMethods

# Required cohort: split the held-out subjects into their own per-subject cohorts so
# fairness/error-rate differences between prosthetic users are visible, not averaged away.
cohorts = [Cohort(name='All test windows (Disabled cohort, held-out subjects)')]
for subject_id in sorted(test_subjects):
    cohort = Cohort(name=f'Held-out subject {subject_id}')
    cohort.add_cohort_filter(CohortFilter(
        method=CohortFilterMethods.METHOD_INCLUDES,
        arg=[subject_id],
        column='subject',
    ))
    cohorts.append(cohort)

ResponsibleAIDashboard(rai_insights, cohort_list=cohorts)


In [ ]:
# Programmatic fallback of the dashboard's fairness/error views, so headline numbers are
# captured in a file even if the interactive widget above is not rendered (e.g. HTML export).
per_subject_rows = []
for subject_id in sorted(test_subjects):
    subset = test_df_full[test_df_full['subject'] == subject_id]
    if subset.empty:
        continue
    predicted = model.predict(subset[feature_columns])
    per_subject_rows.append({
        'subject': subject_id,
        'windows': len(subset),
        'movement_share': float(subset['is_movement'].mean()),
        'accuracy': float(accuracy_score(subset['is_movement'], predicted)),
        'balanced_accuracy': float(balanced_accuracy_score(subset['is_movement'], predicted)),
        'macro_f1': float(f1_score(subset['is_movement'], predicted, average='macro', zero_division=0)),
    })
per_subject_df = pd.DataFrame(per_subject_rows)
print(per_subject_df)

feature_importance = pd.Series(
    np.abs(model.named_steps['logreg'].coef_[0]), index=feature_columns,
).sort_values(ascending=False)
print('\nTop 5 global feature importances (|logistic-regression coefficient| on scaled features):')
print(feature_importance.head(5))

worst_subject = per_subject_df.loc[per_subject_df['balanced_accuracy'].idxmin()] if not per_subject_df.empty else None
results_dir = ROOT / 'results' / 'metrics'
results_dir.mkdir(parents=True, exist_ok=True)
summary = {
    'experiment_id': 'EXP-007-rai-insights-movement-detection',
    'cohort': 'Disabled (Amputee) E1, movement-vs-rest binary gate',
    'seed': SEED,
    'held_out_subjects': sorted(test_subjects),
    'overall_test': {
        'accuracy': float(accuracy_score(test_df_full['is_movement'], test_predictions)),
        'balanced_accuracy': float(balanced_accuracy_score(test_df_full['is_movement'], test_predictions)),
        'macro_f1': float(f1_score(test_df_full['is_movement'], test_predictions, average='macro')),
    },
    'per_subject_test': per_subject_rows,
    'worst_subject': None if worst_subject is None else worst_subject.to_dict(),
    'top_feature_importance': feature_importance.head(5).to_dict(),
    'rai_components_run': ['explainer', 'error_analysis', 'counterfactual', 'causal'],
}
with open(results_dir / 'EXP-007-rai-insights-movement-detection_summary.json', 'w') as handle:
    json.dump(summary, handle, indent=2)
print('\nsaved', results_dir / 'EXP-007-rai-insights-movement-detection_summary.json')


## Non-trivial insights, mitigations, and clinician/engineer-ready recommendations

Fill in the bracketed values after running the cells above; the structure below is what the dashboard's five views are expected to surface for this task, and how each should be read for a prosthetic-control use case.

1. **Model overview and fairness (per-subject cohorts).** Compare `balanced_accuracy` across the per-subject cohorts in the dashboard (or `per_subject_df` above). *So what:* if one held-out subject (`worst_subject` in the exported JSON) scores far below the rest, that is a fairness-relevant, person-specific failure — averaging it into one "overall accuracy" number would hide a person for whom the gate does not work.
2. **Error analysis.** The error-analysis tree/heatmap groups misclassified windows by feature ranges (e.g., low `ch*_rms`). *So what:* if errors concentrate at low EMG amplitude, the gate is least reliable exactly when a user is initiating a movement gently — the highest-risk moment for a prosthetic to stay silent or fire spuriously — so a secondary confirmation step (e.g., require N consecutive movement-classified windows) is a concrete, testable mitigation.
3. **Feature importance.** `feature_importance` ranks which channel/feature (MAV/RMS/WL) the logistic-regression relies on most. *So what:* if importance concentrates on one or two channels, the whole gate is a single-point-of-failure for that electrode — a loose or shifted electrode on those specific channels would silently degrade the controller; recommend redundancy across channels or a minimum-channel-count sensor placement guideline.
4. **Counterfactuals.** DiCE counterfactuals show the smallest EMG-feature change that flips a "Rest" prediction to "Movement" (or vice versa) for a specific window. *So what:* if the counterfactual distance is tiny and physiologically plausible (a small RMS increase), the decision boundary is appropriately sensitive; if it requires an implausible combination of feature changes, that instance's prediction is more fragile/less trustworthy and should be treated with lower confidence in a deployed gate.
5. **Causal analysis.** The estimated effect of `ch1_rms`/`ch1_mav` on `is_movement`, with `subject` as a heterogeneity feature, estimates how much increasing that channel's amplitude changes the predicted movement probability, and whether that effect differs by subject. *So what:* a consistent, subject-independent effect supports the (expected) physiological story that amplitude drives the gate; a strongly heterogeneous effect (different per subject) means a single fixed decision threshold is not appropriate across users and per-user calibration would be needed before deployment — this is a causal/statistical association from observational data, not a proof that channel 1 amplitude physiologically causes the movement label (see `AGENTS.md` "Physiological Interpretation").

### Mitigation plan (draft, to be updated with the actual computed numbers)

- If a specific subject/cohort underperforms: collect a short per-user calibration recording before first use, and recalibrate the threshold per user rather than shipping one global threshold.
- If errors concentrate at low amplitude: add a temporal debounce/hysteresis rule (require sustained movement-classified windows) before actuating, rather than acting on a single-window prediction.
- If importance concentrates on few channels: add a monitoring check for electrode signal quality on those channels specifically, and prefer sensor layouts that spread information across channels.
- If the causal effect is heterogeneous across subjects: do not deploy one global decision threshold; support per-user threshold calibration as part of device fitting.

### Rubric alignment ("So what" summary)

| Component | Rubric link | Prosthetics so-what |
|---|---|---|
| Data analysis | #2 Data characterization | Confirms movement/rest class balance and per-subject data volume before trusting any fairness comparison |
| Model overview & fairness | #9 RAI (fairness) | Surfaces whether the gate is equally reliable across held-out prosthetic users, not just "on average" |
| Error analysis | #9 RAI (error analysis) | Locates exactly which signal regime (e.g., low-amplitude onset) needs a safety mitigation |
| Feature importance | #5, #9 | Identifies single-channel dependency risk for sensor/electrode engineering |
| Counterfactuals | #9 RAI (counterfactuals) | Quantifies how close individual decisions are to flipping, i.e., how fragile a specific prediction is |
| Causal analysis | #9 RAI (causal) | Tests whether one global threshold is defensible or per-user calibration is required |

This remains an offline research analysis on a binary simplification of the movement-recognition task. It does not constitute validation of a real-time, safety-certified prosthetic controller.
